# 2.4 Évaluation Unimodale

Évaluer les 4 méthodes contre les jugements humains.

In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
# Charger les données
ground_truth = pd.read_csv('data/subset_fashion_dataset/ground_truth.csv', sep=';')
df_tfidf = pd.read_csv('results/tfidf_top5_anchors.csv')
df_embeddings = pd.read_csv('results/embeddings_top5_anchors.csv')
df_resnet = pd.read_csv('results/resnet_top5_anchors.csv')
df_vit = pd.read_csv('results/vit_top5_anchors.csv')

print(f"Ancres : {len(ground_truth)}")

Ancres : 30


In [3]:
# Convertir les colonnes top5_ids en listes
import re

def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        # Extraire tous les nombres de la chaîne
        numbers = re.findall(r'\d+', x)
        return [int(n) for n in numbers]
    return []

df_tfidf['top5_ids'] = df_tfidf['top5_ids'].apply(to_list)
df_embeddings['top5_ids'] = df_embeddings['top5_ids'].apply(to_list)
df_resnet['top5_ids'] = df_resnet['top5_ids'].apply(to_list)
df_vit['top5_ids'] = df_vit['top5_ids'].apply(to_list)

print("Listes converties")
print(f"Exemple TF-IDF : {df_tfidf['top5_ids'].iloc[0]}")
print(f"Exemple ResNet : {df_resnet['top5_ids'].iloc[0]}")

Listes converties
Exemple TF-IDF : [41403, 42394, 38808, 39345, 26486]
Exemple ResNet : [64, 4861, 64, 41968, 64, 8356, 64, 40162, 64, 34039]


In [4]:
def evaluate_method(predictions_df, ground_truth_df):
    """Calculer P@5, R@5, MRR pour une méthode"""
    precisions = []
    recalls = []
    mrrs = []
    
    for _, pred_row in predictions_df.iterrows():
        anchor_id = pred_row['anchor_id']
        pred_ids = set(pred_row['top5_ids'])
        
        # Récupérer ground truth
        gt_row = ground_truth_df[ground_truth_df['anchor_id'] == anchor_id]
        if len(gt_row) == 0:
            continue
        gt_row = gt_row.iloc[0]
        
        # Extraire global_ids
        true_ids = set()
        for i in range(1, 6):
            gid = gt_row[f'global_id{i}']
            if pd.notna(gid):
                true_ids.add(int(gid))
        
        # Intersection
        intersection = pred_ids & true_ids
        
        # P@5 et R@5
        precisions.append(len(intersection) / 5)
        recalls.append(len(intersection) / 5)
        
        # MRR
        pred_list = list(pred_row['top5_ids'])
        mrr = 0.0
        for rank, pid in enumerate(pred_list, 1):
            if pid in true_ids:
                mrr = 1.0 / rank
                break
        mrrs.append(mrr)
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'mrr': np.mean(mrrs)
    }

In [5]:
# Évaluer chaque méthode
results = []

metrics_tfidf = evaluate_method(df_tfidf, ground_truth)
results.append({'Méthode': 'TF-IDF', **metrics_tfidf})
print(f"TF-IDF      : P@5={metrics_tfidf['precision']:.3f}, R@5={metrics_tfidf['recall']:.3f}, MRR={metrics_tfidf['mrr']:.3f}")

metrics_emb = evaluate_method(df_embeddings, ground_truth)
results.append({'Méthode': 'Embeddings', **metrics_emb})
print(f"Embeddings  : P@5={metrics_emb['precision']:.3f}, R@5={metrics_emb['recall']:.3f}, MRR={metrics_emb['mrr']:.3f}")

metrics_resnet = evaluate_method(df_resnet, ground_truth)
results.append({'Méthode': 'ResNet-50', **metrics_resnet})
print(f"ResNet-50   : P@5={metrics_resnet['precision']:.3f}, R@5={metrics_resnet['recall']:.3f}, MRR={metrics_resnet['mrr']:.3f}")

metrics_vit = evaluate_method(df_vit, ground_truth)
results.append({'Méthode': 'ViT', **metrics_vit})
print(f"ViT         : P@5={metrics_vit['precision']:.3f}, R@5={metrics_vit['recall']:.3f}, MRR={metrics_vit['mrr']:.3f}")

TF-IDF      : P@5=0.360, R@5=0.360, MRR=0.662
Embeddings  : P@5=0.367, R@5=0.367, MRR=0.661
ResNet-50   : P@5=0.387, R@5=0.387, MRR=0.364
ViT         : P@5=0.447, R@5=0.447, MRR=0.316


In [6]:
# Créer le tableau
df_results = pd.DataFrame(results)
print("\n" + "="*60)
print("TABLEAU COMPARATIF")
print("="*60)
print(df_results.to_string(index=False))
print("="*60)


TABLEAU COMPARATIF
   Méthode  precision   recall      mrr
    TF-IDF   0.360000 0.360000 0.662222
Embeddings   0.366667 0.366667 0.661111
 ResNet-50   0.386667 0.386667 0.364444
       ViT   0.446667 0.446667 0.315833


In [7]:
# Meilleure méthode
best_p = df_results.loc[df_results['precision'].idxmax()]
best_r = df_results.loc[df_results['recall'].idxmax()]
best_mrr = df_results.loc[df_results['mrr'].idxmax()]

print("\nMEILLEURS MÉTHODES :")
print(f"  Precision@5 : {best_p['Méthode']} ({best_p['precision']:.1%})")
print(f"  Recall@5    : {best_r['Méthode']} ({best_r['recall']:.1%})")
print(f"  MRR         : {best_mrr['Méthode']} ({best_mrr['mrr']:.3f})")


MEILLEURS MÉTHODES :
  Precision@5 : ViT (44.7%)
  Recall@5    : ViT (44.7%)
  MRR         : TF-IDF (0.662)


In [8]:
# Sauvegarder
df_results.to_csv('results/evaluation_summary.csv', index=False)
print("\nSauvegardé : results/evaluation_summary.csv")


Sauvegardé : results/evaluation_summary.csv
